# Lab 11 — Loading JSONL with 🤗 Datasets
### Week 2 · Data Engineering for LLM Pipelines

Labs 09–10 produced clean instruction-tuning JSONL. Before any training or evaluation, that
JSONL has to become a **memory-efficient, splittable, tokenized** dataset. This lab uses
**🤗 `datasets`** to load the Lab 10 prompt–completion file (regenerated here so the notebook
stands alone), make **deterministic** train/validation/test splits, **train a tiny tokenizer
offline**, and compare **fixed-length vs dynamic padding**.

**By the end you will be able to:**
1. Load line-delimited **JSONL** into a `datasets.Dataset` / `DatasetDict`.
2. Create **deterministic** splits and shuffle with a fixed seed; persist to Arrow and reload.
3. Apply **batched `map`** transforms (schema alignment, tokenization) and **`filter`**.
4. Contrast **fixed-length** and **dynamic** padding and say which you'd train with.

> **Hints stay light (Labs 05–10).** Each Part opens with a **Toolbox**; you assemble the
> pieces. Target **20/20**; a red check never halts the notebook.
>
> ⚠️ **CURRENCY FLAG — framework format.** We use `datasets` **numpy** format so the lab runs
> anywhere offline. For an actual training run you'd `set_format("torch")` and batch with a
> `torch` `DataLoader` + **`transformers.DataCollatorWithPadding`** (which does the dynamic
> padding you'll hand-roll here). ⚠️ **Offline tokenizer.** No downloads — we train a tiny
> Byte-Level BPE locally with `tokenizers`.


## Setup — offline env, imports, folders, and the `check()` helper

In [ ]:
%pip install -r requirements.txt

In [ ]:

import os
# Force HF offline + local cache BEFORE importing datasets (no network in the classroom).
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import re, json, hashlib, random
from pathlib import Path
from datetime import datetime, timedelta
os.environ.setdefault("HF_HOME", str(Path("artifacts/hf_cache").resolve()))

import numpy as np
import pandas as pd
import orjson
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk

for p in ["artifacts/jsonl", "artifacts/datasets", "artifacts/tokenizer", "artifacts/samples", "artifacts/hf_cache"]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("datasets ready | numpy", np.__version__)

In [ ]:

_score = {"pass": 0, "fail": 0}
def check(label, predicate):
    try:
        ok = bool(predicate() if callable(predicate) else predicate); note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'\u2705 PASS' if ok else '\u274c FAIL'} \u2014 {label}{note}")
def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing  ({_score['fail']} to go)\n{'='*46}")

check("Setup: artifact folders exist", lambda: all(Path(p).is_dir() for p in ["artifacts/datasets","artifacts/tokenizer"]))


### Provided — regenerate the Lab 10 input

In delivery, `instruct_prompt_completion.jsonl` comes from **Lab 10**. We rebuild it here
(same seeds, same pipeline) so this notebook runs standalone: **61** prompt–completion rows.

In [ ]:

def _bootstrap_lab10_pc():
    """Verbatim Lab 09->10 pipeline -> artifacts/jsonl/instruct_prompt_completion.jsonl (61 rows)."""
    rng = random.Random(42)
    TYPES=["help_article","policy","release_note","faq"]; SEC=["Overview","Setup","Troubleshooting","FAQ"]
    TAGS=["billing","security","compliance","sso","api","governance","export","retention","privacy","rate_limits"]
    LANGS=["en","en","en","de","fr"]; CONF=["public","internal"]; now=datetime(2025,2,10)
    boiler=("This article explains how to configure single sign-on with step-by-step instructions. "
            "Use the admin console to enable SAML and verify claim mappings. "
            "Common pitfalls include clock skew and incorrect audience URIs. ")
    rows=[]
    for i in range(1,1001):
        kind=rng.choice(TYPES); doc_id=f"DOC-{i:04d}"
        title={"help_article":f"How to configure SSO (v{rng.randint(1,5)}).",
               "policy":f"Data Retention Policy \u2014 Region {rng.choice(['US','EU','APAC'])}",
               "release_note":f"Release 2025{rng.randint(1,12):02d} \u2014 Key fixes",
               "faq":f"FAQ: {rng.choice(['Exports','Rate Limits','Privacy','Billing'])}"}[kind]
        body=(boiler*rng.randint(1,3))+f"Additional details about {rng.choice(TAGS)} and {rng.choice(TAGS)}. Ref {doc_id}."
        rows.append({"doc_id":doc_id,"type":kind,"title":title,"section":rng.choice(SEC),"body_text":body,
                     "tags":",".join(rng.sample(TAGS,k=rng.randint(2,4))),
                     "source_url":f"https://example.local/{kind}/{doc_id.lower()}",
                     "created_at":(now-timedelta(days=rng.randint(0,240))).strftime('%Y-%m-%d'),
                     "updated_at":(now-timedelta(days=rng.randint(0,30))).strftime('%Y-%m-%d'),
                     "language":rng.choice(LANGS),"confidentiality":rng.choice(CONF)})
    rows[0].update({"confidentiality":"public","language":"en"})
    for j in range(9): rows.append({**rows[0],"doc_id":f"DOC-DUP-{j:02d}"})
    for k in range(5): rows.append({**rows[0],"doc_id":f"DOC-EMPTY-{k:02d}","body_text":""})
    df=pd.DataFrame(rows); nz=lambda s: re.sub(r"\s+"," ",str(s or "")).strip()
    for c in ["title","section","body_text"]: df[c]=df[c].map(nz)
    df=df[(df.confidentiality=="public")&(df.language=="en")]; df=df[df.body_text.str.len()>=30].copy()
    df["ck"]=[hashlib.sha1((nz(t+" "+b)).lower().encode()).hexdigest() for t,b in zip(df.title,df.body_text)]
    df["updated_at"]=pd.to_datetime(df["updated_at"])
    df=df.sort_values(["ck","updated_at"],ascending=[True,False]).drop_duplicates("ck",keep="first")
    sample=df.sample(min(120,len(df)),random_state=7)
    seen=set(); pc=[]
    for r in sample.itertuples():
        instr=f"Summarize the key steps from: {r.title} ({r.section})."
        out="Key steps: enable SAML; map claims; verify time sync; check audience URI; review settings."
        if len(out)<20 or (instr,out) in seen: continue
        seen.add((instr,out))
        pc.append({"prompt":f"### Instruction:\n{instr}\n\n### Response:\n","completion":out,
                   "metadata":{"doc_id":r.doc_id,"schema_version":"trio-v1"}})
    with open("artifacts/jsonl/instruct_prompt_completion.jsonl","wb") as f:
        for r in pc: f.write(orjson.dumps(r)+b"\n")
    return len(pc)

SRC = Path("artifacts/jsonl/instruct_prompt_completion.jsonl")
print("prompt-completion rows:", _bootstrap_lab10_pc())


---
## Part A — Load JSONL with `datasets`

`datasets` memory-maps an Arrow table off disk, so a corpus far larger than RAM loads
instantly and transforms stay lazy until materialized — the reason you reach for it over
`pandas` at corpus scale.

### A1 — Load the file

Load `SRC` as a single-split `Dataset` (the JSON loader's `split="train"` returns the whole
file as one split). Assign it to `ds`.

> **🧰 Toolbox for Part A** — `load_dataset("json", data_files=str(SRC), split="train")` ·
> `ds.num_rows` · `ds.column_names` · `ds.features`.

In [ ]:

ds = load_dataset("json", data_files=str(SRC), split="train")
print("rows:", ds.num_rows, "| columns:", ds.column_names)
ds

In [ ]:

check("A1: 61 rows loaded", lambda: ds.num_rows == 61)
check("A1: prompt + completion columns present with templated prompts",
      lambda: {"prompt", "completion"} <= set(ds.column_names)
              and ds[0]["prompt"].startswith("### Instruction:"))


---
## Part B — Align schema, split deterministically, persist

> **🧰 Toolbox for Part B** — `ds.map(fn, batched=True, remove_columns=...)` ·
> `ds.filter(fn)` · `ds.train_test_split(test_size=..., seed=...)` · `DatasetDict` ·
> `Dataset.shuffle(seed=...)` · `save_to_disk` / `load_from_disk`.


### B1 — Align to `{prompt, completion}` strings

Write a **batched** `ensure_schema(batch)` that returns just `prompt` and `completion` as
strings. Support a Trio fallback: if `prompt`/`completion` are absent but `input`/`output`
are present, template `prompt = "### Instruction:\n{input}\n\n### Response:\n"` and
`completion = output`. Coerce `None`→`""`. Map it over `ds` (dropping the old columns) and
`filter` out rows with an empty prompt or completion → `dsa`.

In [ ]:

def ensure_schema(batch):
    prompts, completions = batch.get("prompt"), batch.get("completion")
    ins, outs = batch.get("input"), batch.get("output")
    if (prompts is None or completions is None) and ins is not None and outs is not None:
        prompts = [f"### Instruction:\n{x}\n\n### Response:\n" for x in ins]
        completions = outs
    return {"prompt": ["" if x is None else str(x) for x in prompts],
            "completion": ["" if x is None else str(x) for x in completions]}

dsa = ds.map(ensure_schema, batched=True, remove_columns=ds.column_names)
dsa = dsa.filter(lambda ex: len(ex["prompt"]) > 0 and len(ex["completion"]) > 0)
print("aligned:", dsa.num_rows, dsa.column_names)

In [ ]:

check("B1: exactly prompt + completion columns", lambda: set(dsa.column_names) == {"prompt", "completion"})
check("B1: 61 rows survive, all strings",
      lambda: dsa.num_rows == 61 and all(isinstance(dsa[i]["prompt"], str) for i in range(min(5, dsa.num_rows))))


### B2 — Deterministic 3-way split

With `seed = 13`: split off **20%** as a holdout, then split that holdout **50/50** into
validation and test. Assemble a `DatasetDict` with keys `train`/`validation`/`test`, and
`shuffle(seed=13)` each split.

> ⚠️ **Determinism is eval integrity.** A fixed seed makes the split reproducible on any
> machine, and the three splits must stay **disjoint** — a row leaking from train into test
> silently inflates your eval score.

In [ ]:

seed = 13
tt = dsa.train_test_split(test_size=0.2, seed=seed)
vt = tt["test"].train_test_split(test_size=0.5, seed=seed)
dds = DatasetDict({"train": tt["train"], "validation": vt["train"], "test": vt["test"]})
dds = DatasetDict({k: v.shuffle(seed=seed) for k, v in dds.items()})
print({k: v.num_rows for k, v in dds.items()})

In [ ]:

def _redo_split():
    a = dsa.train_test_split(test_size=0.2, seed=13)
    b = a["test"].train_test_split(test_size=0.5, seed=13)
    d = DatasetDict({"train": a["train"], "validation": b["train"], "test": b["test"]})
    return DatasetDict({k: v.shuffle(seed=13) for k, v in d.items()})

check("B2: split sizes are train=48, validation=6, test=7",
      lambda: {k: v.num_rows for k, v in dds.items()} == {"train": 48, "validation": 6, "test": 7})
check("B2: split is deterministic (same seed -> identical rows)",
      lambda: all(dds[k]["prompt"] == _redo_split()[k]["prompt"] for k in dds))
check("B2: splits are disjoint (no leakage)",
      lambda: (lambda s: s[0].isdisjoint(s[1]) and s[0].isdisjoint(s[2]) and s[1].isdisjoint(s[2]))(
              [set(dds[k]["prompt"]) for k in ["train","validation","test"]]))


### B3 — Persist to Arrow and reload

`save_to_disk` the whole `DatasetDict` to `artifacts/datasets/instruct_pc_splits`, then
`load_from_disk` it back into `reloaded`.

In [ ]:

SPLIT_DIR = "artifacts/datasets/instruct_pc_splits"
dds.save_to_disk(SPLIT_DIR)
reloaded = load_from_disk(SPLIT_DIR)
print({k: reloaded[k].num_rows for k in reloaded})

In [ ]:

check("B3: reloaded from disk with matching sizes",
      lambda: Path("artifacts/datasets/instruct_pc_splits").exists()
              and {k: reloaded[k].num_rows for k in reloaded} == {"train": 48, "validation": 6, "test": 7})
check("B3: reloaded train content matches (from disk)",
      lambda: Path("artifacts/datasets/instruct_pc_splits").exists()
              and reloaded["train"]["prompt"] == dds["train"]["prompt"])


---
## Part C — Offline tokenizer, then fixed-length tokenization

We train a tiny **Byte-Level BPE** on the training prompts — no downloads. (The training is
**provided**; your job is the encode step.)

In [ ]:

# Provided: train a small offline Byte-Level BPE tokenizer on the training prompts.
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import ByteLevel as ByteLevelProcessor
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

_train_txt = Path("artifacts/tokenizer/train_prompts.txt")
_train_txt.write_text("\n".join(dds["train"]["prompt"]) + "\n", encoding="utf-8")

_tok = Tokenizer(BPE(unk_token="<unk>"))
_tok.pre_tokenizer = ByteLevel()
_tok.train([str(_train_txt)],
           BpeTrainer(vocab_size=8000, min_frequency=2,
                      special_tokens=["<unk>", "<pad>", "<bos>", "<eos>"]))
_tok.post_processor = ByteLevelProcessor()
_tok.decoder = ByteLevelDecoder()
_tok.save("artifacts/tokenizer/bytebpe.json")

tok = Tokenizer.from_file("artifacts/tokenizer/bytebpe.json")
PAD_ID = tok.token_to_id("<pad>")
EOS_ID = tok.token_to_id("<eos>")
MAX_LEN = 128
print("vocab:", tok.get_vocab_size(), "| pad:", PAD_ID, "| eos:", EOS_ID)

In [ ]:

check("C1: tokenizer saved with pad + eos ids", lambda: Path("artifacts/tokenizer/bytebpe.json").exists()
      and PAD_ID is not None and EOS_ID is not None)
check("C1: encode/decode round-trips the prompt text",
      lambda: "Instruction" in tok.decode(tok.encode("### Instruction:\nSummarize").ids))


### C2 — Fixed-length tokenization via batched `map`

Write a **batched** `encode_fixed(batch)` that, for each `prompt`+`completion`: encodes the
concatenation, truncates to `MAX_LEN - 1`, appends `EOS_ID`, then **right-pads with `PAD_ID`
to exactly `MAX_LEN`**. Return `input_ids` and a matching `attention_mask` (`1` for real
tokens incl. eos, `0` for pad). Map it over `dds`.

> **🧰 Toolbox for Part C** — `tok.encode(text).ids` · list slicing · pad with
> `[PAD_ID] * n` · the mask is `1`×(real length) then `0`×(pad length).

In [ ]:

def encode_fixed(batch):
    ids_out, mask_out = [], []
    for p, c in zip(batch["prompt"], batch["completion"]):
        ids = tok.encode(p + c).ids[:MAX_LEN - 1] + [EOS_ID]
        real = len(ids)
        ids = ids + [PAD_ID] * (MAX_LEN - real)
        ids_out.append(ids)
        mask_out.append([1] * real + [0] * (MAX_LEN - real))
    return {"input_ids": ids_out, "attention_mask": mask_out}

enc_fixed = dds.map(encode_fixed, batched=True, batch_size=64,
                    remove_columns=dds["train"].column_names)
_ex = enc_fixed["train"][0]
print("row len:", len(_ex["input_ids"]), "| real tokens:", sum(_ex["attention_mask"]))

In [ ]:

_rows = [enc_fixed["train"][i] for i in range(min(8, enc_fixed["train"].num_rows))]
check("C2: every row padded to exactly MAX_LEN, with real tokens then padding",
      lambda: all(len(r["input_ids"]) == MAX_LEN and 0 < sum(r["attention_mask"]) < MAX_LEN for r in _rows))
check("C2: last real token is EOS (truncate-then-eos worked)",
      lambda: all(0 < sum(r["attention_mask"]) and r["input_ids"][sum(r["attention_mask"]) - 1] == EOS_ID for r in _rows))
check("C2: mask is strictly 1s then 0s",
      lambda: all(0 < sum(r["attention_mask"]) < MAX_LEN and
                  r["attention_mask"] == [1]*sum(r["attention_mask"]) + [0]*(MAX_LEN - sum(r["attention_mask"]))
                  for r in _rows))


---
## Part D — Variable length + dynamic padding, and hygiene

Fixed padding to `MAX_LEN` wastes compute when most rows are short. **Dynamic padding**
tokenizes *without* padding and pads each batch to that batch's longest row.

> **🧰 Toolbox for Part D** — encode without padding (just ids + eos) · `set_format("numpy")`
> · per-batch `max(len(x) for x in batch)` · `np.full` / list padding · `ds.filter` ·
> chained `ds.map`.


### D1 — Variable-length encode + a dynamic-padding collator

First a **batched** `encode_var(batch)` that encodes `prompt`+`completion`, truncates to
`MAX_LEN - 1`, appends `EOS_ID`, and stores the **un-padded** `input_ids` plus its `length`.
Then `collate_dynamic(rows)` that pads a list of rows to the **batch maximum** (not `MAX_LEN`)
and returns `input_ids` / `attention_mask` as 2-D `np.int64` arrays.

In [ ]:

def encode_var(batch):
    ids_out, len_out = [], []
    for p, c in zip(batch["prompt"], batch["completion"]):
        ids = tok.encode(p + c).ids[:MAX_LEN - 1] + [EOS_ID]
        ids_out.append(ids); len_out.append(len(ids))
    return {"input_ids": ids_out, "length": len_out}

enc_var = dds.map(encode_var, batched=True, remove_columns=dds["train"].column_names)

def collate_dynamic(rows):
    width = max(len(r["input_ids"]) for r in rows)          # batch max, not MAX_LEN
    ii = np.full((len(rows), width), PAD_ID, dtype=np.int64)
    am = np.zeros((len(rows), width), dtype=np.int64)
    for i, r in enumerate(rows):
        n = len(r["input_ids"])
        ii[i, :n] = r["input_ids"]; am[i, :n] = 1
    return {"input_ids": ii, "attention_mask": am}

_batch = [enc_var["train"][i] for i in range(min(8, enc_var["train"].num_rows))]
_out = collate_dynamic(_batch)
print("dynamic batch width:", _out["input_ids"].shape[1], "vs MAX_LEN", MAX_LEN)

In [ ]:

_b = [enc_var["train"][i] for i in range(min(8, enc_var["train"].num_rows))]
_o = collate_dynamic(_b)
_wmax = max(len(r["input_ids"]) for r in _b)
check("D1: dynamic width equals the batch's longest row (50 < width < MAX_LEN)",
      lambda: _o["input_ids"].shape[1] == _wmax and 50 < _wmax < MAX_LEN)
check("D1: attention_mask row-sums equal each row's true length",
      lambda: [int(v) for v in _o["attention_mask"].sum(axis=1)] == [len(r["input_ids"]) for r in _b])
check("D1: arrays are rectangular int64 (width > 1)",
      lambda: _o["input_ids"].dtype == np.int64 and _o["input_ids"].shape == _o["attention_mask"].shape
              and _o["input_ids"].shape[1] > 1)


### D2 — Compose a cleaning `map` + a length `filter`

Chain two hygiene steps on `dds`: a batched `strip_ws` map (strip both fields), then a
`filter` keeping only rows whose **prompt** encodes to `≤ 28` tokens. Save the result to
`artifacts/datasets/instruct_pc_clean` and record `n_clean`.

In [ ]:

def strip_ws(batch):
    return {"prompt": [s.strip() for s in batch["prompt"]],
            "completion": [s.strip() for s in batch["completion"]]}

clean = dds.map(strip_ws, batched=True)
clean = clean.filter(lambda ex: len(tok.encode(ex["prompt"]).ids) <= 28)
clean.save_to_disk("artifacts/datasets/instruct_pc_clean")
n_clean = sum(clean[k].num_rows for k in clean)
print("clean rows:", {k: clean[k].num_rows for k in clean}, "total", n_clean)

In [ ]:

_pre = sum(dds[k].num_rows for k in dds)
check("D2: filter kept a strict, non-empty subset with every kept prompt <= 28 tokens and stripped",
      lambda: 0 < n_clean < _pre and all(len(tok.encode(clean["train"][i]["prompt"]).ids) <= 28
              and clean["train"][i]["prompt"] == clean["train"][i]["prompt"].strip()
              for i in range(min(5, clean["train"].num_rows))))
check("D2: cleaned dataset persisted to disk", lambda: Path("artifacts/datasets/instruct_pc_clean").exists())
score()


---
## Wrap-up — answer in this Markdown cell

1. **Why `datasets` over pandas** for a large JSONL corpus — name two concrete reasons.
2. **Reproduce the split** — what seed did you use, and how would a teammate get the *exact*
   same train/validation/test on another machine?
3. **Fixed vs dynamic padding** — which will you train with, and what does dynamic padding
   save here (fixed width `128` vs the dynamic batch width you printed)?

**Key takeaways**
- **Arrow-backed, lazy, memory-mapped.** `datasets` streams from disk and keeps `map`/`filter`
  lazy — that's why it scales past RAM where pandas won't.
- **Determinism is eval integrity.** A fixed seed reproduces the split anywhere; disjoint
  splits stop train/test leakage from inflating your metrics.
- **Batched `map` is the workhorse.** Tokenize with `batched=True` and drop source columns;
  looping row-by-row is the slow anti-pattern.
- **Dynamic padding saves compute.** Pad to the batch max, not a global `MAX_LEN`. In a real
  run you'd let `transformers.DataCollatorWithPadding` do it inside a `torch` `DataLoader`;
  here you hand-rolled the same idea on numpy.
- **Train tokenizers offline** when the classroom has no network — a tiny Byte-Level BPE is
  enough to exercise the whole pipeline.
